# PPO Genetic Photobioreactor - Colab Training
This notebook runs the IBM (Individual-Based Model) simulation for algal growth optimization using PPO.

In [ ]:
# Install Dependencies
!pip install stable-baselines3[extra] shimmy gymnasium

## 1. Define Environment (`genetic_env.py`)

In [ ]:

import gymnasium as gym
from gymnasium import spaces
import numpy as np
from typing import Optional, Tuple, Dict

class GeneticPhotobioreactorEnv(gym.Env):
    """
    Individual-Based Model (IBM) Photobioreactor Environment.
    Tracks N individual algal cells as particles in 1D depth (z-axis) using vectorized operations.
    Implements Genetic Domain Randomization with unique algal strains per episode.
    """
    metadata = {'render_modes': ['human']}

    def __init__(self, max_cells: int = 1000000, initial_cells: int = 500):
        super(GeneticPhotobioreactorEnv, self).__init__()
        
        # --- CONFIGURATION ---
        self.max_cells = max_cells
        self.initial_cells = initial_cells
        self.num_active = initial_cells
        self.dt = 0.01  # Time step (0.01h) determines simulation resolution
        self.reactor_depth = 0.05
        self.volume_L = 1.0
        
        # Action: [Stirring, Light, Nutrient, CO2]
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(4,), dtype=np.float32)
        
        # Obs: [OD, pH, Ext_Nutrients, Dissolved_O2, Temp]
        # Agent sees only macroscopic property averages, not individual cells
        self.observation_space = spaces.Box(
            low=np.array([0, 0, 0, 0, 0]),
            high=np.array([100, 14, 5000, 50, 60]),
            dtype=np.float32
        )
        
        # --- VECTORIZED STATE ARRAYS ---
        # Fixed size arrays, managed via active_mask
        self.cells_z = np.zeros(self.max_cells, dtype=np.float32) 
        self.cells_mass = np.zeros(self.max_cells, dtype=np.float32)
        self.cells_quota = np.zeros(self.max_cells, dtype=np.float32)
        
        # Boolean mask: True = Active Living Cell
        self.active_mask = np.zeros(self.max_cells, dtype=bool)
        
        self.ext_nutrients = 500.0
        self.ph = 7.0
        self.do2 = 6.0
        self.temp = 25.0 # Ambient temperature
        
        # --- GENETIC PARAMS (Placeholder, set in reset) ---
        self.strain_params = {}
        
        self.step_count = 0
        self.max_steps = 15000 # 150h simulated per episode
        # Actually with dt=0.01h, 15000 steps = 150 hours.
        
    def _randomize_strain(self):
        """Generates a unique 'Strain' of algae for this episode."""
        self.strain_params = {
            'mu_max': np.random.normal(0.25, 0.05),
            'Ks': np.random.normal(20.0, 5.0),
            'Ki': np.random.normal(120.0, 30.0),
            'Kii': np.random.normal(2000.0, 500.0),
            'T_opt': np.random.normal(27.0, 2.0), # Optimal Temp
            'Q_min': 1.5,
            'Q_max': 5.0
        }
        self.strain_params['mu_max'] = max(0.05, self.strain_params['mu_max'])
        self.strain_params['Ki'] = max(10, self.strain_params['Ki'])
        self.strain_params['T_opt'] = np.clip(self.strain_params['T_opt'], 20.0, 35.0)

    def reset(self, seed: Optional[int] = None, options: Optional[Dict] = None):
        super().reset(seed=seed)
        self._randomize_strain()
        
        # Initialize Population
        self.num_active = self.initial_cells
        self.active_mask[:] = False
        self.active_mask[:self.num_active] = True
        
        self.cells_z[:self.num_active] = np.random.uniform(0, self.reactor_depth, self.num_active)
        self.cells_mass[:self.num_active] = np.random.normal(50.0, 5.0, self.num_active)
        self.cells_quota[:self.num_active] = np.random.uniform(2.0, 4.0, self.num_active)
        
        self.ext_nutrients = 200.0
        self.ph = 7.5
        self.do2 = 7.0
        self.temp = 25.0
        self.step_count = 0
        
        return self._get_obs(), {}

    def _get_obs(self):
        # Calculate stats only on active cells
        if self.num_active > 0:
            total_mass = np.sum(self.cells_mass[self.active_mask])
        else:
            total_mass = 0.0
            
        od_proxy = total_mass / (2000.0 * 50.0 * 2.0) # Relative to baseline capacity 
        
        base_obs = np.array([
            od_proxy,
            self.ph,
            self.ext_nutrients,
            self.do2,
            self.temp
        ], dtype=np.float32)

        # Stochastic Sensor Noise: +/- 1%
        noise_factor = np.random.uniform(0.99, 1.01, size=base_obs.shape)
        noisy_obs = base_obs * noise_factor
        
        return noisy_obs.astype(np.float32)

    def step(self, action):
        stir_act, light_act, nut_act, co2_act = action
        
        stir_rpm = np.interp(stir_act, [-1, 1], [0, 500])
        I_surface = np.interp(light_act, [-1, 1], [0, 2000])
        nut_flow = np.interp(nut_act, [-1, 1], [0, 10])
        co2_flow = np.interp(co2_act, [-1, 1], [0, 1])
        
        # --- Safety Checks ---
        # 0. Check for invalid actions
        if np.any(np.isnan(action)):
            # If action is NaN (e.g. model output is corrupted), default to 0
            stir_rpm = 0.0; I_surface = 0.0; nut_flow = 0.0; co2_flow = 0.0

        # --- Physics (Langevin Dynamics) ---
        # Apply only to active cells
        Diffusion = 1e-6 + (stir_rpm * 1e-5)
        dt_sec = self.dt * 3600
        
        # Generate noise for everyone (easier than masking noise generation)
        noise = np.random.normal(0, 1, self.max_cells)
        dz = np.sqrt(2 * Diffusion * dt_sec) * noise
        
        # Update Z masked
        self.cells_z[self.active_mask] += dz[self.active_mask]
        
        # Boundary Conditions
        # Note: Vectorized logical ops on masked arrays can be tricky, 
        # so we modify the whole array but valid data is only at active_mask.
        self.cells_z = np.abs(self.cells_z)
        over_bottom = self.cells_z > self.reactor_depth
        self.cells_z[over_bottom] = 2*self.reactor_depth - self.cells_z[over_bottom]
        self.cells_z = np.clip(self.cells_z, 0, self.reactor_depth)

        # --- Biology ---
        if self.num_active > 0:
            n_spawns = 0
            params = self.strain_params
            
            # 1. Light Field (Self-Shading)
            od = self._get_obs()[0]
            # Safety clamp for OD
            od = np.nan_to_num(od, nan=0.0, posinf=100.0)
            
            k_ext = 0.2 + (20.0 * od) 
            # prevent overflow in exp if z is huge (unlikely but safe)
            exponent = -k_ext * self.cells_z
            exponent = np.clip(exponent, -50, 0) # e^-50 is basically 0
            cells_I = I_surface * np.exp(exponent)
            
            # 2. Temperature Factor (Gaussian)
            # T_opt randomization test
            temp_factor = np.exp(-0.5 * ((self.temp - params['T_opt'])/5.0)**2)
            
            # 3. Growth Rate
            f_I = cells_I / (params['Ki'] + cells_I + (cells_I**2 / params['Kii']))
            f_I = np.nan_to_num(f_I)
            
            # Droop Quota
            # Only update active quotas
            current_quotas = self.cells_quota[self.active_mask]
            
            f_Q = np.maximum(0.0, 1.0 - params['Q_min'] / (current_quotas + 1e-6))
            
            # Clamp mu to prevent explosion
            current_mu = params['mu_max'] * f_I[self.active_mask] * f_Q * temp_factor
            current_mu = np.clip(current_mu, 0.0, 5.0) 
            
            # --- Maintenance Respiration ---
            # Cells have a base metabolic cost (approx 5% of max growth)
            m_respiration = 0.05 * params['mu_max']
            
            # Net Growth Rate = Photosynthesis - Respiration
            # This can be negative (mass loss) if light/nutrients are insufficient!
            net_mu = current_mu - m_respiration
            
            # Grow Biomass
            growth_mult = np.exp(net_mu * self.dt)
            # Clip multiplier to avoid single-step explosion (both up and down)
            growth_mult = np.clip(growth_mult, 0.5, 2.0)
            
            self.cells_mass[self.active_mask] *= growth_mult
            
            # Safeguard: Hard cap on cell mass 
            self.cells_mass[self.active_mask] = np.clip(self.cells_mass[self.active_mask], 1e-3, 1e6)
            
            # --- DEATH (Starvation) ---
            # If net_mu is negative long enough, mass drops.
            # Kill cells that fall below 10 pg (Starvation Threshold)
            
            # We map back to full array
            dead_indices = np.where(self.active_mask & (self.cells_mass < 10.0))[0]
            n_deaths = len(dead_indices)
            
            if n_deaths > 0:
                self.active_mask[dead_indices] = False
                self.num_active -= n_deaths
                # Optional: Zero out dead memory for cleanliness
                self.cells_mass[dead_indices] = 0.0
                self.cells_quota[dead_indices] = 0.0
                
            # Nutrient Uptake
            uptake_rate = 0.5 * (self.ext_nutrients / (params['Ks'] + self.ext_nutrients))
            uptake_amount = uptake_rate * self.dt
            
            # Update Quota & External Nutrients
            self.cells_quota[self.active_mask] += uptake_amount
            
            total_uptake_mg = (np.sum(uptake_amount) * self.num_active) * 0.001
            self.ext_nutrients = max(0, self.ext_nutrients - total_uptake_mg + nut_flow)
            
            # --- CELL DIVISION (Reproduction) ---
            # Threshold: 100pg (Double start mass)
            ready_to_divide = (self.active_mask) & (self.cells_mass > 100.0)
            n_dividing = np.sum(ready_to_divide)
            
            if n_dividing > 0:
                # Find empty slots
                inactive_indices = np.where(~self.active_mask)[0]
                n_slots = len(inactive_indices)
                n_spawns = min(n_dividing, n_slots)
                
                if n_spawns > 0:
                    # Indices of parents that get to spawn (capped by slots)
                    # We need precise mapping. 
                    parent_indices = np.where(ready_to_divide)[0][:n_spawns]
                    child_indices = inactive_indices[:n_spawns]
                    
                    # Split Mass
                    self.cells_mass[parent_indices] *= 0.5
                    
                    # Activate Children
                    self.active_mask[child_indices] = True
                    self.cells_mass[child_indices] = self.cells_mass[parent_indices] # Start with half mass
                    self.cells_z[child_indices] = self.cells_z[parent_indices]     # Inherit position
                    self.cells_quota[child_indices] = self.cells_quota[parent_indices] # Inherit quota status
                    
                    self.num_active += n_spawns
            
        else:
            n_spawns = 0
            current_mu = np.zeros(1) # fallback

        # --- Environmental Dynamics (Macro) ---

        if self.step_count % 100 == 0:
            # Rebalance array occasionally if needed (not strictly necessary with this mask implementation)
            pass

        # --- Environmental Dynamics (Macro) ---

        # 1. Shear Stress (RPM > 400)
        # Random death probability for cells if mixing is too violent
        if stir_rpm > 400.0:
            shear_risk = (stir_rpm - 400.0) / 100.0 # 0.0 to 1.0
            prob_death = 0.01 * shear_risk * self.dt # 1% per hour at max RPM
            
            # Apply to active cells
            survival_mask = np.random.uniform(0, 1, self.num_active) > prob_death
            
            # We need to map this reduced mask back to the full active_mask
            # Get indices of currently active cells
            curr_active_indices = np.where(self.active_mask)[0]
            # Identify which ones died
            dying_indices = curr_active_indices[~survival_mask]
            
            if len(dying_indices) > 0:
                self.active_mask[dying_indices] = False
                self.num_active -= len(dying_indices)
                self.cells_mass[dying_indices] = 0.0
                self.cells_quota[dying_indices] = 0.0

        # 2. Gas Exchange (O2 & CO2)
        # k_La determines how fast gases exchange with air.
        # Stirring increases surface area and turbulence -> Higher k_La
        # Base transfer + Mixing enhancement
        k_La = 0.5 + (4.0 * (stir_rpm / 500.0)) # range 0.5 to 4.5 /hr
        
        # Dissolved Oxygen Dynamics
        # Production: Proportional to Growth (approx 1.5g O2 per g Biomass)
        # Respiration: Proportional to maintenance (approx 1.0g O2 per g Biomass lost)
        # Calculate net biomass change from biology step (approx)
        total_biomass = np.sum(self.cells_mass[self.active_mask]) * 1e-9 # mg -> kg? No, mass is pg. 1pg = 1e-12g.
        # Let's use relative units. Total Mass in mg.
        total_mass_mg = np.sum(self.cells_mass[self.active_mask]) * 1e-9 # pg * 1e-9 = mg? 
        # 1 pg = 10^-12 g. 1 mg = 10^-3 g. So 1 pg = 10^-9 mg. Correct.

        # Simplify: Delta Mass roughly tracks O2. 
        if self.step_count > 0:
            delta_mass_mg = total_mass_mg - (self.last_mass * 1e-9)
        else:
            delta_mass_mg = 0.0
            
        o2_production = delta_mass_mg * 1.2 # mg O2 produced
        
        # Gas Transfer
        # O2 Saturation at 25C is approx 8.0 mg/L
        o2_sat = 8.0 
        o2_transfer = k_La * (o2_sat - self.do2) * self.dt
        
        self.do2 += (o2_production / self.volume_L) + o2_transfer
        self.do2 = np.clip(self.do2, 0.0, 30.0) # Cap at realistic supersaturation
        
        # 3. pH Dynamics (CO2)
        # Simplification:
        # - Photosynthesis (Growth) -> Consumes CO2 -> pH Rises
        # - Respiration (Loss) -> Releases CO2 -> pH Drops
        # - CO2 Injection -> Adds CO2 -> pH Drops dramatically
        # - Air Exchange -> Tends toward 7.5 (Atmospheric equilibrium)
        
        # Efficiency of CO2 injection depends on Mixing!
        co2_dissolution_eff = 0.1 + (0.9 * (stir_rpm / 500.0))
        
        ph_rise = (delta_mass_mg * 0.1) # Growth consumes Carbon
        ph_drop_inject = co2_flow * 2.0 * co2_dissolution_eff * self.dt # Injection
        ph_restore = k_La * (7.5 - self.ph) * 0.1 * self.dt # Air exchange drift
        
        self.ph += ph_rise - ph_drop_inject + ph_restore
        self.ph = np.clip(self.ph, 4.0, 10.0)

        # 4. Thermal Mass Balance
        # Temp_new = Temp_old + (Light * Heat_Factor) - (Ambient_Loss)
        # Light Source: 2000 umol/m2/s ~ 400 W/m2 PAR ~ heat input
        heat_factor = 2.0 # deg C per hour at max light
        cooling_coeff = 0.5 # /hr exchange with room air
        
        heat_gain = (I_surface / 2000.0) * heat_factor * self.dt
        heat_loss = cooling_coeff * (self.temp - 25.0) * self.dt
        
        self.temp += heat_gain - heat_loss
        
        # --- Reward ---
        total_mass = np.sum(self.cells_mass[self.active_mask]) if self.num_active > 0 else 0
        if self.step_count == 0: self.last_mass = total_mass
        
        # Productivity + Reproduction Bonus
        # Productivity + Reproduction Bonus
        productivity = total_mass - self.last_mass
        # Handle NaN/Inf in productivity
        if not np.isfinite(productivity):
            productivity = 0.0
            
        self.last_mass = total_mass
        
        reward = (productivity * 1.0) + (n_spawns * 5.0) # strong bonus for division
        
        # Final safeguard on reward
        if not np.isfinite(reward):
            reward = -10.0 # Punishment for breaking physics
        
        # Penalty for crashing population
        if self.num_active < 10:
            reward -= 10.0
            done = True
        else:
            self.step_count += 1
            done = self.step_count >= self.max_steps
        
        # Debug Print
        if self.step_count % 100 == 0:
             print(f"[EnvDebug] Step: {self.step_count}, Active: {self.num_active}, Done: {done}")
             
        return self._get_obs(), float(reward), done, False, {"pop": self.num_active}



## 2. Train Agent (`ppo_genetic.py`)

In [ ]:

import gymnasium as gym
import numpy as np
import os
import time

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.callbacks import BaseCallback
import pickle


class GeneticMonitorCallback(BaseCallback):
    """
    Custom callback to log the Genetic Parameters of each episode.
    """
    def __init__(self, verbose=0):
        super(GeneticMonitorCallback, self).__init__(verbose)
        self.episode_count = 0

    def _on_step(self) -> bool:
        # Detect new episodes to log the new random strain
        dones = self.locals['dones']
        if dones[0]:
            self.episode_count += 1
            env = self.training_env.envs[0]
            params = env.strain_params
            
            summary = f"Episode {self.episode_count} Finished. Next Strain: " \
                      f"Mu={params['mu_max']:.2f}, Ki={params['Ki']:.0f}"
            print(f"[GeneticMonitor] {summary}")
            
        return True

def main():
    print("--- PPO Agent for Genetic IBM Photobioreactor ---")
    print("Initializing Vectorized Individual-Based Model (2000 cells)...")
    
    # Create Environment
    env = DummyVecEnv([lambda: GeneticPhotobioreactorEnv(max_cells=1000000, initial_cells=500)])
    
    # Apply VecNormalize to handle large rewards and unscaled observations
    env = VecNormalize(env, norm_obs=True, norm_reward=True, clip_obs=100.0)
    
    model_name = "ppo_genetic_ibm"
    tensorboard_log = "./ppo_genetic_tensorboard/"
    
    model = PPO(
        "MlpPolicy",
        env,
        verbose=1,
        learning_rate=3e-4,
        n_steps=2048,
        batch_size=64,
        n_epochs=10,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        tensorboard_log=tensorboard_log,
        device="auto" # Auto-detect (CUDA if available, else CPU)
    )
    
    print("Starting Training with Genetic Domain Randomization...")
    print("Agent will face a different 'Algal Strain' every episode.")
    
    model.learn(
        total_timesteps=300_000, # Increased for better convergence
        callback=GeneticMonitorCallback(),
        progress_bar=True
    )
    
    print("Training Complete.")
    model.save(model_name)
    env.save("vec_normalize.pkl") # Save normalization stats
    print(f"Model saved to {model_name}.zip")
    print("Normalization stats saved to vec_normalize.pkl")

    # --- Verification Run ---
    print("\n--- Verifying Robustness on 3 New Strains ---")
    obs = env.reset()
    for i in range(3):
        print(f"\nTest Run {i+1} (New Random Strain)...")
        # Get params from internal env
        # Note: input_env[0] is the way to access un-vec env
        current_params = env.envs[0].strain_params 
        print(f"Strain Params: {current_params}")
        
        total_reward = 0
        done = False
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, dones, info = env.step(action)
            total_reward += reward[0]
            done = dones[0]
            
        print(f"Total Reward: {total_reward:.2f}")

if __name__ == "__main__":
    main()


## 3. Download Model
Download the trained model and normalization statistics immediately after training.

In [ ]:
try:
    from google.colab import files
    import os

    if os.path.exists('ppo_genetic_ibm.zip'):
        files.download('ppo_genetic_ibm.zip')
    else:
        print('Model file not found!')

    if os.path.exists('vec_normalize.pkl'):
        files.download('vec_normalize.pkl')
    else:
        print('Normalization stats not found!')
except ImportError:
    print('Not running in Google Colab, skipping download.')

## 4. Evaluate Agent (`evaluate_agent.py`)

In [ ]:

import gymnasium as gym
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
import pandas as pd

class Evaluator:
    def __init__(self, model_path="ppo_genetic_ibm", stats_path="vec_normalize.pkl"):
        # 1. Create base env
        self.env = DummyVecEnv([lambda: GeneticPhotobioreactorEnv(max_cells=500000, initial_cells=500)])
        
        # 2. Wrap with VecNormalize (must match training structure)
        # We start with default args, then load stats
        self.env = VecNormalize(self.env, norm_obs=True, norm_reward=False, clip_obs=100.0)
        
        # 3. Load Normalization Statistics
        try:
            self.env = VecNormalize.load(stats_path, self.env.venv)
            # IMPORTANT: Disable training updates and reward normalization during eval
            self.env.training = False
            self.env.norm_reward = False
            print(f"Normalization stats loaded from {stats_path}")
        except FileNotFoundError:
            print(f"Stats {stats_path} not found! EVALUATION MAY BE INCORRECT if model was trained with normalization.")

        try:
            self.model = PPO.load(model_path, env=self.env)
            print(f"Model loaded from {model_path}")
        except FileNotFoundError:
            print(f"Model {model_path} not found! Please train first.")
            self.model = None

    def evaluate_episode(self, agent_type="ppo"):
        """
        Runs a single episode and returns stats.
        agent_type: 'ppo' or 'random'
        """
        obs = self.env.reset()
        
        # Get the random strain params for this episode
        # Note: In DummyVecEnv, envs are a list.
        strain_params = self.env.envs[0].strain_params
        
        total_reward = 0
        done = False
        step = 0
        
        while not done:
            if agent_type == "ppo" and self.model:
                action, _ = self.model.predict(obs, deterministic=True)
            else:
                action = [self.env.action_space.sample()]
                
            obs, reward, dones, info = self.env.step(action)
            done = dones[0] # Extract scalar boolean from list
            total_reward += reward[0] # VecEnv returns list
            step += 1
            
            # Print progress every 100 steps
            if step % 100 == 0:
                print(f"    Step {step}...", end="\r")
            
            # Safety Break
            if step > 16000:
                print(f"\n[WARN] Episode exceeded 11000 steps! Force breaking.")
                break
            
        # Get final population from info dict (since VecEnv resets env on done)
        if 'pop' in info[0]:
            final_pop = info[0]['pop']
        else:
            final_pop = self.env.envs[0].num_active
        
        return {
            "Agent": agent_type,
            "Strain_T_opt": strain_params['T_opt'],
            "Strain_Mu": strain_params['mu_max'],
            "Total_Reward": total_reward,
            "Final_Pop": final_pop,
            "Steps": step
        }

    def run_benchmark(self, num_episodes=5):
        results = []
        
        print(f"\n--- Running Benchmark ({num_episodes} eps per agent) ---")
        
        # 1. PPO Evaluation
        if self.model:
            print("Evaluating PPO Agent...")
            for i in range(num_episodes):
                res = self.evaluate_episode("ppo")
                results.append(res)
                print(f"  PPO Ep {i+1}: Reward={res['Total_Reward']:.1f}, Pop={res['Final_Pop']}, T_opt={res['Strain_T_opt']:.1f}")

        # 2. Random Evaluation
        print("Evaluating Random Baseline...")
        for i in range(num_episodes):
            res = self.evaluate_episode("random")
            results.append(res)
            print(f"  RND Ep {i+1}: Reward={res['Total_Reward']:.1f}, Pop={res['Final_Pop']}, T_opt={res['Strain_T_opt']:.1f}")

        # Analysis
        df = pd.DataFrame(results)
        print("\n--- Summary ---")
        print(df.groupby("Agent")[["Total_Reward", "Final_Pop"]].mean())
        
        return df

if __name__ == "__main__":
    evaluator = Evaluator()
    evaluator.run_benchmark(num_episodes=5)
